In [16]:
import numpy as np
import axelrod
import skfuzzy as fuzz
from skfuzzy import control as ctrl
from axelrod.action import Action, actions_to_str
from axelrod.player import Player
from axelrod.strategy_transformers import (
    FinalTransformer,
    TrackHistoryTransformer,
)
import random
from copy import deepcopy
from collections import Counter


In [17]:
PARAM_KEYS = [
    'D_b', 'D_c',
    'C_a', 'C_b',
    'd_threshold', 'c_threshold',
]

BOUNDS = [
    (1,  49),   # D_b
    (1,  60),   # D_c
    (25, 74),   # C_a
    (25, 99),   # C_b
    (0.1, 0.8), # d_threshold
    (0.2, 0.9), # c_threshold
]

YOUR_BASELINE = [
    25, 50,   # D_b, D_c
    35, 75,   # C_a, C_b
    0.4, 0.6, # thresholds
]

In [18]:
def decode(individual):
    """Convert flat list back to named params dict."""
    return dict(zip(PARAM_KEYS, individual))

def repair(individual):
    ind = individual.copy()

    # Clip to bounds
    for i, (lo, hi) in enumerate(BOUNDS):
        ind[i] = float(np.clip(ind[i], lo, hi))

    # D_b <= D_c
    ind[1] = max(ind[0], ind[1])

    # C_a <= C_b
    ind[3] = max(ind[2], ind[3])

    return ind

In [19]:
C, D = Action.C, Action.D

class FuzzyMethods():
    @staticmethod
    def calcCooperation(self, opponent):
        return (Counter(opponent.history)[C])/len(opponent.history)*100
    
    @staticmethod
    def calcAdaptivity(self, opponent):
        adapCounter = 0
        adapReaction = 0

        if(len(self.history) < 3): 
            return 0
        
        for i in range(3, len(self.history)):
            if (self.history[i-3] == C and self.history[i-2] == D):
                adapCounter += 1
                if (opponent.history[i-1] == D):
                    adapReaction += 1
                elif (self.history[i-1] == D and opponent.history[i] == D):
                    adapReaction += 0.5
            elif (self.history[i-3] == D and self.history[i-2] == C):
                adapCounter += 1
                if (opponent.history[i-1] == C):
                    adapReaction += 1
                elif (self.history[i-1] == C and opponent.history[i] == C):
                    adapReaction += 0.5

        if adapCounter == 0:
            return 0
        
        return adapReaction/adapCounter*100

    
    @staticmethod
    def calcForgiveness(self, opponent):
        DCounter = 0
        punishmentCounter = 0

        for i in range(0, len(self.history)-1):
            if(self.history[i] == D):
                DCounter += 1
                for j in range (i+1, len(opponent.history)):
                    if(opponent.history[j] == C):
                        break
                    else:
                        punishmentCounter += 1
              

        if punishmentCounter > 0:
            return DCounter/punishmentCounter*100
        else:
            return 100
    
    @staticmethod
    def calcStochastic(self, opponent):
        patterns = [
            [C, C, C],
            [C, C, D],
            [C, D, C],
            [C, D, D],
            [D, C, C],
            [D, C, D],
            [D, D, C],
            [D, D, D]
        ]

        non_stochasticCounter = 0
        patternPlayedCounter = 0

        for p in patterns:
            opponentsReactions = []
            for i in range(0, len(self.history)-3):
                if ([self.history[i], self.history[i+1], self.history[i+2]] == p):
                    opponentsReactions.append([opponent.history[i+1], opponent.history[i+2], opponent.history[i+3]])
            
            unique_patterns = len(set(tuple(sub) for sub in opponentsReactions))

            if(len(opponentsReactions) > 0):
                non_stochasticCounter += 0 if unique_patterns == 1 else unique_patterns
                patternPlayedCounter += len(opponentsReactions)
        
        if patternPlayedCounter == 0:
            return 0
        
        return non_stochasticCounter/patternPlayedCounter*100
    

In [20]:
def build_player(params):
    """Build a fresh OptimizedFuzzy player from params dict."""

    _cooperation = ctrl.Antecedent(np.arange(0, 100, 1), 'cooperation')
    _adaptivity  = ctrl.Antecedent(np.arange(0, 100, 1), 'adaptivity')
    _forgiveness = ctrl.Antecedent(np.arange(0, 100, 1), 'forgiveness')
    _stochastic  = ctrl.Antecedent(np.arange(0, 100, 1), 'stochastic')

    _cooperation.automf(names=["low", "medium", "high"])
    _adaptivity.automf(names=["no", "yes"])
    _forgiveness.automf(names=["low", "medium", "high"])
    _forgiveness['low'] = fuzz.gaussmf(_forgiveness.universe, 0, 25)
    _forgiveness['medium'] = fuzz.trimf(_forgiveness.universe, [25, 50, 75])
    _stochastic.automf(names=["none", "sometimes", "always"])

    _resulting_strategy = ctrl.Consequent(np.arange(0, 100, 1), 'resulting_strategy')
    _resulting_strategy['D'] = fuzz.trimf(_resulting_strategy.universe, [0, params['D_b'], params['D_c']])
    _resulting_strategy['C'] = fuzz.trimf(_resulting_strategy.universe, [params['C_a'], params['C_b'], 99])

    rule_default = ctrl.Rule(_cooperation['low'] | _cooperation['medium'] | _cooperation['high'], _resulting_strategy['C'])
    rule1 = ctrl.Rule(_cooperation['high'] & _adaptivity['no'] & (_forgiveness['medium'] | _forgiveness['high']), _resulting_strategy['D'])
    rule2 = ctrl.Rule(_forgiveness['low'] & _cooperation['high'], _resulting_strategy['C'])
    rule3 = ctrl.Rule(_stochastic['always'] | _adaptivity['no'], _resulting_strategy['D'])
    rule4 = ctrl.Rule(_cooperation['low'] | (_cooperation['medium'] & _forgiveness['low']), _resulting_strategy['D'])
    rule5 = ctrl.Rule(_cooperation['medium'] & _forgiveness['medium'] & _adaptivity['yes'], _resulting_strategy['C'])

    _chosen_strategy = ctrl.ControlSystemSimulation(ctrl.ControlSystem([rule_default, rule1, rule2, rule3, rule4, rule5]))

    class GAFuzzy(Player):
        name = "GAFuzzy"
        classifier = {"memory_depth": float("inf"), "stochastic": False,
                      "long_run_time": False, "inspects_source": False,
                      "manipulates_source": False, "manipulates_state": False}

        cooperation        = _cooperation
        adaptivity         = _adaptivity
        forgiveness        = _forgiveness
        stochastic         = _stochastic
        resulting_strategy = _resulting_strategy
        chosen_strategy    = _chosen_strategy
        d_thresh           = params['d_threshold']
        c_thresh           = params['c_threshold']
        first_time         = True
        h                  = {'Name': '', 'Fuzzy': [], 'Opponent': []}

        def strategy(self, opponent: Player) -> Action:
            if len(self.history) == 0 or D not in opponent.history:
                return C
            coop  = FuzzyMethods.calcCooperation(self, opponent)
            adap  = FuzzyMethods.calcAdaptivity(self, opponent)
            forg  = FuzzyMethods.calcForgiveness(self, opponent)
            stoch = FuzzyMethods.calcStochastic(self, opponent)
            self.chosen_strategy.input['cooperation'] = coop
            self.chosen_strategy.input['adaptivity']  = adap
            self.chosen_strategy.input['forgiveness'] = forg
            self.chosen_strategy.input['stochastic']  = stoch
            try:
                self.chosen_strategy.compute()
                output_val   = self.chosen_strategy.output['resulting_strategy']
                d_membership = fuzz.interp_membership(self.resulting_strategy.universe, self.resulting_strategy['D'].mf, output_val)
                c_membership = fuzz.interp_membership(self.resulting_strategy.universe, self.resulting_strategy['C'].mf, output_val)
                if d_membership >= self.d_thresh and c_membership < self.c_thresh:
                    return D
            except:
                return C
            return C

    return GAFuzzy()



In [21]:
def build_player_from_individual(individual):
    ind = repair(individual)
    p   = decode(ind)
    params = {
        'D_a': 0,             'D_b': int(p['D_b']), 'D_c': int(p['D_c']),
        'C_a': int(p['C_a']), 'C_b': int(p['C_b']), 'C_c': 99,
        'd_threshold': p['d_threshold'],
        'c_threshold': p['c_threshold'],
    }
    return build_player(params)

In [22]:
def evaluate(individual):
    try:
        fuzzy_player = build_player_from_individual(individual)
        opponents    = [s() for s in axelrod.stewart_plotkin_strategies]
        results      = axelrod.Tournament([fuzzy_player] + opponents, turns=200, repetitions=3).play(progress_bar=False)
        return np.mean(results.normalised_scores[0])
    except Exception as e:
        print(f"  [evaluate ERROR] {type(e).__name__}: {e}")
        return 0.0

In [23]:
def random_individual():
    return [random.uniform(lo, hi) for lo, hi in BOUNDS]


def crossover(parent1, parent2):
    """Blend crossover (BLX-alpha)."""
    alpha  = 0.3
    child1 = []
    child2 = []
    for a, b in zip(parent1, parent2):
        lo   = min(a, b) - alpha * abs(a - b)
        hi   = max(a, b) + alpha * abs(a - b)
        child1.append(random.uniform(lo, hi))
        child2.append(random.uniform(lo, hi))
    return repair(child1), repair(child2)


def mutate(individual, mutation_rate=0.15, mutation_strength=0.1):
    """Gaussian mutation — each gene mutates independently."""
    mutant = individual.copy()
    for i, (lo, hi) in enumerate(BOUNDS):
        if random.random() < mutation_rate:
            sigma        = (hi - lo) * mutation_strength
            mutant[i]   += random.gauss(0, sigma)
    return repair(mutant)


def tournament_selection(population, fitnesses, k=3):
    """Pick k random individuals, return the best."""
    candidates = random.sample(list(zip(population, fitnesses)), k)
    return max(candidates, key=lambda x: x[1])[0]


# ─────────────────────────────────────────────
#  MAIN GA LOOP
# ─────────────────────────────────────────────

def run_ga(
    pop_size        = 30,
    n_generations   = 50,
    crossover_prob  = 0.8,
    mutation_rate   = 0.15,
    elitism         = 2,       # how many top individuals carry over unchanged
):
    print("Initialising population...")

    # Seed population with your baseline so generation 0 already has a strong individual
    population  = [YOUR_BASELINE.copy()]
    population += [random_individual() for _ in range(pop_size - 1)]

    best_individual = None
    best_score      = -1

    for gen in range(n_generations):

        # Evaluate all individuals
        fitnesses = [evaluate(ind) for ind in population]

        # Track best
        gen_best_idx   = int(np.argmax(fitnesses))
        gen_best_score = fitnesses[gen_best_idx]

        if gen_best_score > best_score:
            best_score      = gen_best_score
            best_individual = population[gen_best_idx].copy()

        print(f"Gen {gen+1:>3}/{n_generations} | Best: {gen_best_score:.4f} | Pop avg: {np.mean(fitnesses):.4f} | All-time best: {best_score:.4f}")

        # Elitism — carry top individuals forward unchanged
        sorted_pop = [x for _, x in sorted(zip(fitnesses, population), key=lambda p: p[0], reverse=True)]
        new_population = sorted_pop[:elitism]

        # Fill rest of population with offspring
        while len(new_population) < pop_size:
            parent1 = tournament_selection(population, fitnesses)
            parent2 = tournament_selection(population, fitnesses)

            if random.random() < crossover_prob:
                child1, child2 = crossover(parent1, parent2)
            else:
                child1, child2 = parent1.copy(), parent2.copy()

            new_population.append(mutate(child1, mutation_rate))
            if len(new_population) < pop_size:
                new_population.append(mutate(child2, mutation_rate))

        population = new_population

    print(f"\n=== GA COMPLETE ===")
    print(f"Best score: {best_score:.4f}")
    print(f"Best params: {decode(best_individual)}")
    return best_individual, best_score


if __name__ == "__main__":
    best_ind, best_score = run_ga()

Initialising population...
Gen   1/50 | Best: 2.6143 | Pop avg: 2.5987 | All-time best: 2.6143
Gen   2/50 | Best: 2.6218 | Pop avg: 2.5995 | All-time best: 2.6218
Gen   3/50 | Best: 2.6186 | Pop avg: 2.6032 | All-time best: 2.6218
Gen   4/50 | Best: 2.6250 | Pop avg: 2.6022 | All-time best: 2.6250
Gen   5/50 | Best: 2.6204 | Pop avg: 2.6021 | All-time best: 2.6250
Gen   6/50 | Best: 2.6189 | Pop avg: 2.6028 | All-time best: 2.6250
Gen   7/50 | Best: 2.6307 | Pop avg: 2.6072 | All-time best: 2.6307
Gen   8/50 | Best: 2.6186 | Pop avg: 2.6040 | All-time best: 2.6307
Gen   9/50 | Best: 2.6157 | Pop avg: 2.6037 | All-time best: 2.6307
Gen  10/50 | Best: 2.6239 | Pop avg: 2.6045 | All-time best: 2.6307
Gen  11/50 | Best: 2.6236 | Pop avg: 2.6083 | All-time best: 2.6307
Gen  12/50 | Best: 2.6214 | Pop avg: 2.6051 | All-time best: 2.6307
Gen  13/50 | Best: 2.6207 | Pop avg: 2.6026 | All-time best: 2.6307
Gen  14/50 | Best: 2.6196 | Pop avg: 2.6025 | All-time best: 2.6307
Gen  15/50 | Best: 2.